# TripoSR — Single Image to 3D

Independent Colab/T4 reconstruction baseline. Requires a GPU runtime and internet access for weights. Upload a single object image, export GLB, validate the mesh and download it.

Uses pretrained `VAST-AI-Research/TripoSR` (model `stabilityai/TripoSR`). We deliberately replace `torchmcubes` with scikit-image CPU marching cubes because the CUDA extension failed to compile in the tested Colab environment. This is a disclosed local modification, and output must be visually inspected.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime"
print(torch.cuda.get_device_name(0))

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.10"], check=True)

In [ ]:
import subprocess
from pathlib import Path

env = Path("/content/triposr-env/bin/python")
if not env.exists():
    subprocess.run(["uv", "venv", "/content/triposr-env", "--python", "3.10"], check=True)
print("TripoSR Python:", env.exists())

In [ ]:
import subprocess
from pathlib import Path
repo = Path("/content/TripoSR")
if not (repo / "run.py").is_file():
    if repo.exists():
        raise RuntimeError("TripoSR directory exists without run.py; inspect it before cloning")
    subprocess.run(["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git", str(repo)], check=True)
print("TripoSR:", (repo / "run.py").exists())

In [ ]:
import subprocess
subprocess.run([
    "uv", "pip", "install", "--python", "/content/triposr-env/bin/python",
    "torch==2.1.2", "torchvision==0.16.2",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

In [ ]:
import subprocess
subprocess.run([
    "/content/triposr-env/bin/python", "-c",
    "import torch; assert torch.cuda.is_available(), 'GPU unavailable'; print(torch.__version__, torch.cuda.get_device_name(0))"
], check=True)

In [ ]:
import subprocess
from pathlib import Path

lines = Path("/content/TripoSR/requirements.txt").read_text().splitlines()
lines = [line for line in lines if "torchmcubes" not in line and line.strip() != "gradio" and not line.startswith("huggingface-hub")]
lines += [
    "huggingface-hub==0.17.3",
    "numpy==1.26.4",
    "scikit-image==0.22.0",
    "onnxruntime-gpu"
]
Path("/content/triposr-req.txt").write_text("\n".join(lines) + "\n")
subprocess.run([
    "uv", "pip", "install", "--python", "/content/triposr-env/bin/python", "-r", "/content/triposr-req.txt"
], check=True)

In [ ]:
from pathlib import Path

path = Path("/content/TripoSR/tsr/models/isosurface.py")
source = path.read_text()

old = "from torchmcubes import marching_cubes"

replacement = """
from skimage.measure import marching_cubes as _skimage_mc

def marching_cubes(volume, level):
    verts, faces, _, _ = _skimage_mc(
        volume.detach().float().cpu().numpy(),
        level=level
    )

    verts = np.ascontiguousarray(verts[:, ::-1])
    faces = np.ascontiguousarray(faces.astype(np.int64))

    return torch.from_numpy(verts), torch.from_numpy(faces)
"""

if old in source:
    path.write_text(source.replace(old, replacement))
elif "_skimage_mc" not in source:
    raise RuntimeError("Unexpected isosurface.py content")

print("Marching cubes patched")

In [ ]:
import subprocess

subprocess.run([
    "/content/triposr-env/bin/python",
    "-c",
    """
import numpy as np
import torch
from tsr.models.isosurface import marching_cubes

x, y, z = np.indices((16, 16, 16))
volume = ((x-8)**2 + (y-8)**2 + (z-8)**2 - 25).astype(np.float32)

vertices, faces = marching_cubes(torch.from_numpy(volume), 0.0)

assert len(vertices) > 0
assert len(faces) > 0

print("Vertices:", len(vertices))
print("Faces:", len(faces))
print("Mesh extraction passed")
"""
], cwd="/content/TripoSR", check=True)

## Upload a test image and reconstruct

Use a centered single object photo or ControlNet-generated image. Reconstruction with `--foreground-ratio 0.95` gave a better chair backrest in the previous comparison.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
assert uploaded, "Upload an object image"
image_file = Path("/content") / next(iter(uploaded))
print(image_file)

In [ ]:
import subprocess
subprocess.run([
    "/content/triposr-env/bin/python", "run.py", str(image_file),
    "--output-dir", "/content/triposr-output",
    "--model-save-format", "glb",
    "--foreground-ratio", "0.95",
    "--mc-resolution", "192",
    "--chunk-size", "4096"
], cwd="/content/TripoSR", check=True)

In [ ]:
from pathlib import Path
import subprocess

mesh_path = Path("/content/triposr-output/0/mesh.glb")
final_path = Path("/content/triposr-output/model.glb")
assert mesh_path.is_file(), "TripoSR output missing"

subprocess.run([
    "/content/triposr-env/bin/python", "-c",
    f"""
import numpy as np
import trimesh
s = trimesh.load({str(mesh_path)!r}, force='scene')
assert s.geometry and s.bounds is not None
for mesh in s.geometry.values():
    if mesh.is_watertight and mesh.volume < 0:
        mesh.invert()
s.export({str(final_path)!r})
v = trimesh.load({str(final_path)!r}, force='scene')
assert v.geometry and np.isfinite(v.bounds).all() and np.all(v.extents > 0)
print('Meshes:', len(v.geometry), 'Extents:', v.extents)
"""
], check=True)
print(final_path)

In [ ]:
from google.colab import files
files.download("/content/triposr-output/model.glb")